# Analyse de la dérive — données de production

**Question** : les clients servis par l'API ressemblent-ils à ceux d'une période de
référence ? Sinon, quel effet sur les décisions du modèle ?

Le journal Supabase ne stocke que l'identifiant du client. Le magasin étant figé et
versionné, la jointure restitue exactement les 145 features soumises au modèle.

1. Manifestes · 2. Lecture du journal · 3. Reconstitution des features ·
4. Déclaration des colonnes · 5. Témoin négatif · 6. Témoin positif ·
7. Effet sur la sortie du modèle · 8. Interprétation

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset
from supabase import create_client

# Le notebook peut être lancé depuis notebooks/ ou depuis la racine.
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CAMPAGNES = RACINE / "data" / "campagnes"
MAGASIN = RACINE / "data" / "store.parquet"
TABLE = "journal_predictions"
ARTEFACTS = RACINE / "artifacts"
RAPPORTS = RACINE / "rapports"

load_dotenv(RACINE / ".env")
URL, CLE = os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"]

print(RACINE)

/Users/fatih/Desktop/AI ENGINEER/Projet_8_Confirmez_vos_competences_en_MLOps/home-credit-scoring-api


## 1. Les manifestes

La table n'a pas de colonne « campagne » : la requête ne porte qu'un identifiant. Les
campagnes se séparent par les bornes temporelles que le script d'injection écrit dans
ses manifestes.

In [2]:
fichiers = sorted(CAMPAGNES.glob("*.json"))
manifestes = {}

for fichier in fichiers:
    manifeste = json.loads(fichier.read_text())
    manifestes[manifeste["campagne"]] = manifeste
    print(
        f"{manifeste['campagne']:<10} {manifeste['debut']} -> {manifeste['fin']}\n"
        f"{'':<10} {manifeste['reponses']}  age median {manifeste['age_median_ans']} ans"
    )

# Un manifeste d'essai oublie fausserait tout : mieux vaut echouer ici.
assert len(fichiers) == 2, f"{len(fichiers)} manifestes, 2 attendus"
assert set(manifestes) == {"reference", "derive"}

derive     2026-09-19T17:24:37.234027+00:00 -> 2026-09-19T17:41:28.755980+00:00
           {'200': 2000}  age median 29.1 ans
reference  2026-09-19T17:06:31.747735+00:00 -> 2026-09-19T17:23:26.485805+00:00
           {'200': 2000}  age median 43.1 ans


## 2. Lecture du journal

L'API REST de Supabase renvoie au plus 1 000 lignes par requête : lecture par pages,
triées sur `id` pour qu'aucune ligne ne soit lue deux fois ni oubliée.

In [3]:
TAILLE_PAGE = 1000
client = create_client(URL, CLE)


def lire_fenetre(debut: str, fin: str) -> pd.DataFrame:
    lignes, depart = [], 0
    while True:
        page = (
            client.table(TABLE)
            .select("*")
            .gte("horodatage", debut)
            .lte("horodatage", fin)
            .order("id")
            .range(depart, depart + TAILLE_PAGE - 1)
            .execute()
            .data
        )
        lignes += page
        if len(page) < TAILLE_PAGE:
            return pd.DataFrame(lignes)
        depart += TAILLE_PAGE

In [4]:
journaux = {
    nom: lire_fenetre(m["debut"], m["fin"]) for nom, m in manifestes.items()
}

for nom, journal in journaux.items():
    attendu = manifestes[nom]["reponses"].get("200", 0)
    print(f"{nom:<10} {len(journal):>5} lignes lues, {attendu:>5} appels en 200")
    print(f"{'':<10} statuts : {journal['statut'].value_counts().to_dict()}")

derive      2001 lignes lues,  2000 appels en 200
           statuts : {200: 2001}
reference   2001 lignes lues,  2000 appels en 200
           statuts : {200: 2001}


Chaque campagne compte une ligne de plus que d'appels : l'appel de réveil du Space,
journalisé sur le premier client de la campagne. Il est retiré ci-dessous.

In [5]:
# keep="last" conserve l'appel de campagne : le tri sur id suit l'ordre d'insertion,
# et le reveil s'ecrit en premier.
journaux = {
    nom: journal.drop_duplicates("sk_id_curr", keep="last")
    for nom, journal in journaux.items()
}

for nom, journal in journaux.items():
    assert len(journal) == manifestes[nom]["reponses"]["200"], nom
    print(f"{nom:<10} {len(journal)} lignes")

derive     2000 lignes
reference  2000 lignes


## 3. Reconstitution des features servies

Jointure des identifiants journalisés au magasin.

In [6]:
magasin = pd.read_parquet(MAGASIN)
print(f"magasin : {magasin.shape[0]} clients, {magasin.shape[1]} features")

features = {}
for nom, journal in journaux.items():
    # Les 404 n'ont pas d'identifiant exploitable et le modele n'a pas tourne.
    servis = journal.loc[journal["statut"] == 200, "sk_id_curr"].astype("int64")
    features[nom] = magasin.loc[servis]
    print(f"{nom:<10} {features[nom].shape}")

magasin : 307505 clients, 145 features
derive     (2000, 145)
reference  (2000, 145)


In [7]:
# Controle du biais : l'age doit separer nettement les deux campagnes.
JOURS_PAR_AN = 365.25

ages = pd.DataFrame(
    {
        "magasin": (-magasin["DAYS_BIRTH"] / JOURS_PAR_AN).describe(),
        "reference": (-features["reference"]["DAYS_BIRTH"] / JOURS_PAR_AN).describe(),
        "derive": (-features["derive"]["DAYS_BIRTH"] / JOURS_PAR_AN).describe(),
    }
).round(1)

ages

,magasin,reference,derive
count,307505.0,2000.0,2000.0
mean,43.9,44.1,28.8
std,11.9,11.8,3.3
min,20.5,21.0,21.1
25%,34.0,34.5,26.6
50%,43.1,43.1,29.1
75%,53.9,53.9,31.5
max,69.1,68.6,34.0


**Contrôle.** `reference` doit épouser `magasin` : le tirage est uniforme. `derive` doit
être nettement plus jeune, avec un maximum vers 34 ans : le biais a pris.

## 4. Déclaration des colonnes

Evidently choisit sa méthode selon le type déclaré de chaque colonne. La liste vient des
artefacts du modèle : la même qui a servi à l'entraînement.

In [8]:
noms = json.loads((ARTEFACTS / "feature_names.json").read_text())
categorielles = list(json.loads((ARTEFACTS / "categories.json").read_text()))
numeriques = [c for c in noms if c not in categorielles]

assert len(noms) == 145
assert set(categorielles) <= set(noms)

schema = DataDefinition(numerical_columns=numeriques, categorical_columns=categorielles)
RAPPORTS.mkdir(exist_ok=True)

print(f"{len(numeriques)} numériques, {len(categorielles)} catégorielles")

129 numériques, 16 catégorielles


In [9]:
def comparer(courant: pd.DataFrame, reference: pd.DataFrame, nom: str):
    # Ordre impose par Evidently 0.7 : courant d'abord, reference ensuite.
    # L'inverse tournerait sans erreur et produirait un rapport a l'envers.
    resultat = Report([DataDriftPreset()]).run(
        Dataset.from_pandas(courant.reset_index(drop=True), data_definition=schema),
        Dataset.from_pandas(reference.reset_index(drop=True), data_definition=schema),
    )
    resultat.save_html(str(RAPPORTS / f"{nom}.html"))
    return resultat

## 5. Témoin négatif — deux tirages du même magasin

Deux échantillons disjoints de 2 000 clients, tirés au hasard dans le magasin. Rien ne
les distingue : les colonnes signalées ici mesurent le **bruit de fond** du détecteur.
Le témoin positif ne se lira qu'au-dessus de ce bruit.

In [10]:
TAILLE, GRAINE = 2000, 42

# sample tire sans remise : les deux moities n'ont aucun client en commun.
tirage = magasin.sample(2 * TAILLE, random_state=GRAINE)
temoin_a, temoin_b = tirage.iloc[:TAILLE], tirage.iloc[TAILLE:]
print(temoin_a.shape, temoin_b.shape)

(2000, 145) (2000, 145)


In [11]:
temoin = comparer(temoin_b, temoin_a, "temoin_negatif")

![Rapport Evidently du témoin négatif](../captures/temoin_negatif.png)

Rapport complet : `rapports/temoin_negatif.html`, régénéré à chaque exécution.

## 6. Témoin positif — la campagne jeunes

La campagne dérivée ne tire que des clients de 34 ans ou moins : une dérive provoquée,
dont la cause est connue. Elle est comparée à la campagne de référence, comme un système
de monitoring compare la période courante à une période stable.

In [12]:
positif = comparer(features["derive"], features["reference"], "derive_campagne_jeunes")

![Rapport Evidently du témoin positif](../captures/temoin_positif.png)

Rapport complet : `rapports/derive_campagne_jeunes.html`.

### Deux familles de scores

Evidently choisit sa méthode colonne par colonne, après retrait des valeurs vides : un
**test** si la référence garde 1 000 valeurs ou moins, une **distance** au-delà
(documentation d'Evidently, page *Data drift*). Les deux se lisent en sens opposé :

- distance (Wasserstein, Jensen-Shannon) : dérive si **≥ 0,1** ;
- p-value (K-S, khi-deux, test Z) : dérive si **≤ 0,05**.

Le rapport HTML trie tous les scores ensemble : des p-values proches de 1, qui signifient
*aucune dérive*, y apparaissent en tête. Les cellules suivantes appliquent à chaque
famille sa propre règle.

In [13]:
def derive_par_colonne(resultat) -> pd.DataFrame:
    table = pd.DataFrame(
        [
            {
                "colonne": m["config"]["column"],
                "methode": m["config"]["method"],
                "score": m["value"],
                "seuil": m["config"]["threshold"],
            }
            for m in resultat.dict()["metrics"]
            if m["config"]["type"].endswith("ValueDrift")
        ]
    )
    table["p_value"] = table["methode"].str.contains("p_value")
    table["derive"] = np.where(
        table["p_value"], table["score"] <= table["seuil"], table["score"] >= table["seuil"]
    )
    return table


def decompte_evidently(resultat) -> dict:
    return next(
        m["value"]
        for m in resultat.dict()["metrics"]
        if m["config"]["type"].endswith("DriftedColumnsCount")
    )


rapports = {"négatif": temoin, "positif": positif}
colonnes = {nom: derive_par_colonne(resultat) for nom, resultat in rapports.items()}

In [14]:
bilan = pd.DataFrame(
    {
        nom: {
            "colonnes en dérive": int(colonnes[nom]["derive"].sum()),
            "décompte Evidently": int(decompte_evidently(resultat)["count"]),
            "part (%)": round(100 * colonnes[nom]["derive"].mean(), 1),
        }
        for nom, resultat in rapports.items()
    }
)

# Les deux premieres lignes doivent etre egales : c'est la preuve que la regle de
# decision reproduit celle d'Evidently.
bilan

,négatif,positif
colonnes en dérive,2.0,29.0
décompte Evidently,2.0,29.0
part (%),1.4,20.0


### Les colonnes qui dérivent

Les deux familles se lisent séparément : une distance et une p-value ne se comparent pas.

In [15]:
table_positif = colonnes["positif"]

(
    table_positif[table_positif["derive"] & ~table_positif["p_value"]]
    .sort_values("score", ascending=False)
    [["colonne", "methode", "score"]]
    .reset_index(drop=True)
)

,colonne,methode,score
0,DAYS_BIRTH,Wasserstein distance (normed),1.295053
1,BUREAU_DAYS_CREDIT_min,Wasserstein distance (normed),0.481903
2,DAYS_ID_PUBLISH,Wasserstein distance (normed),0.440977
3,DAYS_EMPLOYED,Wasserstein distance (normed),0.431824
4,BUREAU_DAYS_CREDIT_mean,Wasserstein distance (normed),0.409284
5,DAYS_REGISTRATION,Wasserstein distance (normed),0.335275
6,EXT_SOURCE_3,Wasserstein distance (normed),0.326858
7,ORGANIZATION_TYPE,Jensen-Shannon distance,0.263209
8,NAME_INCOME_TYPE,Jensen-Shannon distance,0.239319
9,FLAG_EMP_PHONE,Jensen-Shannon distance,0.238861


In [16]:
(
    table_positif[table_positif["derive"] & table_positif["p_value"]]
    .sort_values("score")
    [["colonne", "score"]]
    .reset_index(drop=True)
)

,colonne,score
0,EXT_SOURCE_1,6.622275e-44
1,BUREAU_BB_STATUS_0_mean,3.783939e-03
2,BUREAU_BB_STATUS_C_mean,1.094348e-02


### Au-dessus du bruit de fond

Une colonne qui dérive aussi dans le témoin négatif peut s'expliquer par le hasard.
Celles qui dérivent **seulement** dans le témoin positif sont celles que la campagne
jeunes a déplacées.

In [17]:
derives = {nom: set(table.loc[table["derive"], "colonne"]) for nom, table in colonnes.items()}

print("positif seulement :", sorted(derives["positif"] - derives["négatif"]))
print("positif et négatif :", sorted(derives["positif"] & derives["négatif"]))

table_temoin = colonnes["négatif"]
table_temoin[table_temoin["derive"]][["colonne", "methode", "score"]]

positif seulement : ['AMT_CREDIT', 'AMT_GOODS_PRICE', 'BUREAU_BB_STATUS_0_mean', 'BUREAU_BB_STATUS_C_mean', 'BUREAU_CREDIT_COUNT', 'BUREAU_DAYS_CREDIT_mean', 'BUREAU_DAYS_CREDIT_min', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_REGISTRATION', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'FLAG_DOCUMENT_6', 'FLAG_EMP_PHONE', 'HOUR_APPR_PROCESS_START', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'PREV_AMT_APPLICATION_mean', 'PREV_AMT_CREDIT_mean', 'PREV_POS_DPD_DEF_mean', 'REG_CITY_NOT_WORK_CITY']
positif et négatif : []


,colonne,methode,score
109,BUREAU_AMT_OVERDUE_mean,Wasserstein distance (normed),0.419502
110,BUREAU_DAYS_OVERDUE_max,Wasserstein distance (normed),0.121003


## 7. L'effet sur la sortie du modèle

Le signal le plus direct : score et décision sont dans le journal, sans besoin des
features.

In [18]:
sorties = pd.DataFrame(
    {
        nom: {
            "score moyen": journal["score_risque"].mean(),
            "score médian": journal["score_risque"].median(),
            "taux de refus (%)": 100 * (journal["decision"] == "REFUSE").mean(),
        }
        for nom, journal in journaux.items()
    }
).round(3)

sorties

,derive,reference
score moyen,0.485,0.398
score médian,0.493,0.363
taux de refus (%),49.250,32.350


In [19]:
schema_score = DataDefinition(numerical_columns=["score_risque"])

derive_scores = Report([DataDriftPreset()]).run(
    Dataset.from_pandas(journaux["derive"][["score_risque"]].reset_index(drop=True), data_definition=schema_score),
    Dataset.from_pandas(journaux["reference"][["score_risque"]].reset_index(drop=True), data_definition=schema_score),
)
derive_scores.save_html(str(RAPPORTS / "derive_scores.html"))
derive_par_colonne(derive_scores)[["colonne", "methode", "score", "seuil", "derive"]]

,colonne,methode,score,seuil,derive
0,score_risque,Wasserstein distance (normed),0.404151,0.1,True


## 8. Interprétation

**Le détecteur est fiable dans les deux sens.** Deux tirages du même magasin ne font
apparaître que 2 colonnes en dérive sur 145 (1,4 %). La campagne jeunes en fait
apparaître 29 (20 %), dont aucune n'est commune avec le témoin négatif : elles ne
relèvent pas du bruit de fond.

**Les deux alertes du témoin négatif** portent sur des retards de paiement au bureau de
crédit (`BUREAU_AMT_OVERDUE_mean`, `BUREAU_DAYS_OVERDUE_max`). Leur cause n'est pas
établie. Elles n'occupent que les rangs 63 et 108 sur 145 de l'importance par permutation
du modèle (`artifacts/permutation_importance.csv`) : leur effet sur le score est
négligeable.

**La dérive du témoin positif découle de l'âge.** `DAYS_BIRTH`, la variable biaisée,
présente la plus forte distance (1,30). Les autres colonnes en dérive décrivent des
caractéristiques qui évoluent avec l'âge : ancienneté dans l'emploi, de la pièce
d'identité et de l'historique au bureau de crédit, situation familiale, type de revenu.
Ce lien relève du raisonnement, non d'une mesure.

**Le score augmente parce que les features les plus influentes dérivent.** Sept des huit
features les plus importantes du modèle sont en dérive : `EXT_SOURCE_2`, `EXT_SOURCE_3`,
`ORGANIZATION_TYPE`, `EXT_SOURCE_1`, `AMT_CREDIT`, `AMT_GOODS_PRICE` et `DAYS_BIRTH`.
Le score moyen passe de 0,398 à 0,485 et le taux de refus de 32,35 % à 49,25 % : le
modèle juge la population jeune plus risquée.

**Ce que l'analyse ne dit pas.** Sans défauts observés en production, rien ne permet de
savoir si ces refus supplémentaires sont justifiés : une dérive signale un changement de
population, non une baisse de performance. Entraîné avec `is_unbalance=True`, le modèle
ne produit pas des probabilités de défaut ; ses scores se comparent d'une période à
l'autre, ils ne se lisent pas comme un risque.

**Limites.** Dérive provoquée et de cause connue ; 2 000 clients par période ; seuils par
défaut d'Evidently. Sur les colonnes peu renseignées, le test dispose de moins de valeurs
et repère moins bien un écart modéré.

**En production.** La distribution des scores est le premier signal à surveiller : elle
ne demande que le journal. La dérive des features sert ensuite au diagnostic. La décision
de réentraîner attend les défauts réels.